In [1]:
import sys
# Force reinstall of kagglehub and kagglesdk
!{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir kagglehub kagglesdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 102.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 238.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 283.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 187.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 305.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 399.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 169.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 353.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 366.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 422.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 388.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Ejercicio 7: Bases de Datos Vectoriales

Daniel Flores

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber usar las herramientas actuales.


### Actividad

1. Carga el corpus


In [2]:
# =====================================================
# CELDA DE CARGA DEL CORPUS (NUEVA VERSIÓN SIN TOKEN)
# =====================================================

# 1. Limpiamos los paquetes problemáticos y usamos una versión estable de kagglehub
!pip uninstall -y kagglehub kagglesdk
!pip install kagglehub==0.3.0 pandas --quiet

import kagglehub
import pandas as pd
import os

# 2. Descargamos el dataset (es público, no pide token)
print("🚀 Descargando el dataset de Wikipedia... (esto puede tardar un poco)")
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")
print(f"✅ Dataset descargado correctamente en: {path}")

# 3. Vemos qué archivos hay dentro
archivos = os.listdir(path)
print(f"\n📂 Archivos encontrados dentro del dataset: {archivos}")

# 4. Seleccionamos el primer archivo y lo cargamos como DataFrame
nombre_archivo = archivos[0]  # Normalmente es el único archivo CSV
ruta_completa = os.path.join(path, nombre_archivo)
df = pd.read_csv(ruta_completa)

print("\n🎉 ¡Corpus cargado exitosamente! Primeras 5 filas:")
print(df.head())

Found existing installation: kagglehub 1.0.2
Uninstalling kagglehub-1.0.2:
  Successfully uninstalled kagglehub-1.0.2
Found existing installation: kagglesdk 0.1.33
Uninstalling kagglesdk-0.1.33:
  Successfully uninstalled kagglesdk-0.1.33
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
🚀 Descargando el dataset de Wikipedia... (esto puede tardar un poco)


100%|██████████| 18.7M/18.7M [00:00<00:00, 78.1MB/s]

Extracting model files...


✅ Dataset descargado correctamente en: /root/.cache/kagglehub/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects/versions/1

📂 Archivos encontrados dentro del dataset: ['wikipedia_text_corpus.csv']

🎉 ¡Corpus cargado exitosamente! Primeras 5 filas:
   Unnamed: 0                                               text
0           1  Anovo\n\nAnovo (formerly A Novo) is a computer...
1           2  Battery indicator\n\nA battery indicator (also...
2           3  Bob Pease\n\nRobert Allen Pease (August 22, 19...
3           4  CAVNET\n\nCAVNET was a secure military forum w...
4           5  CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [4]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [5]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [6]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [7]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [8]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [9]:
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 96.3 MB/s eta 0:00:00


In [10]:
import faiss
import numpy as np

# 1. Creo el índice. IndexFlatIP = producto interno (equivale a coseno si los vectores están normalizados).
index_faiss = faiss.IndexFlatIP(embeddings.shape[1])

# 2. Cargo los embeddings del corpus (N x D)
index_faiss.add(embeddings)

# 3. Busco los 10 más parecidos a mi query
D, I = index_faiss.search(query_vec, k=10)

# D = scores de similitud, I = índices dentro de embeddings
print(f"Query: {query_text}\n")
for rank, (idx, score) in enumerate(zip(I[0], D[0]), start=1):
    texto = chunks_df.iloc[idx]["text"]
    print(f"[{rank}] score={score:.4f} :: {texto[:150].strip()}...")

Query: Battery measuring

[1] score=0.8703 :: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing...
[2] score=0.8618 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
[3] score=0.8401 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
[4] score=0.8391 :: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the...
[5] score=0.8386 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approximately...
[6] score=0.8345 :: shorting the measurement points together and performing an adjustment for zero ohms in

### Explicación del bloque FAISS

- `IndexFlatIP(D)`: crea un índice plano (fuerza bruta) usando producto interno (Inner Product). Como los vectores de E5 vienen normalizados, el producto interno da lo mismo que la similitud coseno. Si usara `IndexFlatL2` sería distancia euclidiana.
- `index_faiss.add(embeddings)`: mete los vectores. FAISS los guarda tal cual en RAM.
- `index_faiss.search(query_vec, k=10)`: devuelve dos matrices. `D` con los scores (una fila por query, K columnas). `I` con los índices de los vecinos.
- Con `I[0]` accedo a los índices de la primera (y única) query. Ese índice mapea a la fila de `chunks_df`.

*Dato curioso de álgebra lineal:* si dos vectores tienen norma 1, entonces `dot(a,b) = cos(θ)`. Por eso se puede usar `IndexFlatIP` como sinónimo de coseno cuando las embeddings vienen normalizadas. Un truco común para ahorrar cálculo.

FAISS es rapidísimo pero no guarda metadata. Solo vectores. Para asociar texto o filtros toca gestionarlo por fuera, como acabo de hacer con `chunks_df.iloc[idx]`. Ahí es donde entran Qdrant, Milvus, etc.

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [11]:
!pip install qdrant-client --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 12.0 MB/s eta 0:00:00


In [12]:
!pip install --upgrade qdrant-client --force-reinstall --no-cache-dir --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 267.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 202.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 204.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 147.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 249.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 242.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 237.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 226.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 199.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 169.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [13]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Cliente en memoria (no necesita servidor)
qdrant = QdrantClient(":memory:")

D = embeddings.shape[1]  # dimensión de los vectores

# 1. Creo la colección con métrica coseno
qdrant.recreate_collection(
    collection_name="wiki",
    vectors_config=VectorParams(size=D, distance=Distance.COSINE),
)

# 2. Preparo los puntos (id + vector + payload con el texto y metadata)
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "text": chunks_df.iloc[i]["text"],
            "doc_id": int(chunks_df.iloc[i]["doc_id"]),
            "chunk_id": int(chunks_df.iloc[i]["chunk_id"]),
        }
    )
    for i in range(len(embeddings))
]

# 3. Inserto en lotes para no saturar memoria
BATCH = 1000
for i in range(0, len(points), BATCH):
    qdrant.upsert(collection_name="wiki", points=points[i:i+BATCH])

print(f"Qdrant listo con {len(points)} vectores")

/tmp/ipykernel_7019/3934382746.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(
/tmp/ipykernel_7019/3934382746.py:32: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant.upsert(collection_name="wiki", points=points[i:i+BATCH])


Qdrant listo con 79104 vectores


In [28]:
def qdrant_search(query_embedding, k=5):
    """Retorna lista de (id, score, text, metadata)."""
    # Aseguramos que el vector sea una lista
    vector = query_embedding[0].tolist() if hasattr(query_embedding[0], 'tolist') else query_embedding[0]

    # Usamos query_points que es el método recomendado en versiones recientes
    res = qdrant.query_points(
        collection_name="wiki",
        query=vector,
        limit=k,
    )
    return [
        (h.id, h.score, h.payload["text"], {"doc_id": h.payload["doc_id"], "chunk_id": h.payload["chunk_id"]})
        for h in res.points
    ]

# Ejemplo con k=5
try:
    resultados = qdrant_search(query_vec, k=5)
    print(f"Query: {query_text}\n")
    for rank, (idx, score, text, meta) in enumerate(resultados, 1):
        print(f"[{rank}] id={idx} score={score:.4f} :: {text[:150].strip()}...")
except Exception as e:
    print(f"Error al buscar: {e}")

Query: Battery measuring

[1] id=10176 score=0.8703 :: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing...
[2] id=1 score=0.8618 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
[3] id=10177 score=0.8401 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
[4] id=37406 score=0.8391 :: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the...
[5] id=71872 score=0.8386 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approximately...


### Respuestas a las preguntas de Qdrant

**¿Cosine o L2? ¿Por qué?** Cosine. Los embeddings de E5 vienen normalizados a norma 1, así que coseno y producto interno dan lo mismo. Y coseno es lo estándar para retrieval de texto porque mide dirección semántica, ignora el largo del documento.

**¿Filtrar por metadata vs FAISS?** Muchísimo más fácil. En FAISS toca traer los índices y filtrar en pandas por fuera. En Qdrant se pasa un `Filter` dentro del `search` y la base solo devuelve lo que cumple. Ejemplo: `qdrant.search(..., query_filter=Filter(must=[FieldCondition(key="doc_id", match=MatchValue(value=42))]))`.

**¿Tiempo al aumentar k?** Con `IndexFlat` (fuerza bruta) el tiempo casi no cambia con k porque el cuello de botella es comparar contra todos los N vectores. K solo afecta al ordenamiento final. Con índices ANN sí puede subir un poco.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [15]:
!pip install "pymilvus[milvus_lite]" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 30.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


In [16]:
from pymilvus import MilvusClient
import os

# Milvus Lite: guarda todo en un archivo local, no necesita servidor
DB_FILE = "./milvus_wiki.db"
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

milvus = MilvusClient(DB_FILE)

# Creo la colección con el índice ANN por defecto (AUTOINDEX = HNSW en Milvus Lite)
milvus.create_collection(
    collection_name="wiki",
    dimension=embeddings.shape[1],
    metric_type="COSINE",
)

# Preparo los datos: cada fila es un dict con id, vector y metadata
data = [
    {
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": chunks_df.iloc[i]["text"],
        "doc_id": int(chunks_df.iloc[i]["doc_id"]),
    }
    for i in range(len(embeddings))
]

# Inserto en lotes
BATCH = 1000
for i in range(0, len(data), BATCH):
    milvus.insert(collection_name="wiki", data=data[i:i+BATCH])

print(f"Milvus listo con {len(data)} vectores")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Milvus listo con 79104 vectores


In [17]:
import time

def milvus_search(query_embedding, k=5):
    res = milvus.search(
        collection_name="wiki",
        data=[query_embedding[0].tolist()],
        limit=k,
        output_fields=["text", "doc_id"],
    )
    hits = res[0]
    return [(h["id"], h["distance"], h["entity"]["text"], {"doc_id": h["entity"]["doc_id"]}) for h in hits]

# Mini experimento: k=5 vs k=20 (tiempo y resultados)
for k in [5, 20]:
    t0 = time.time()
    res = milvus_search(query_vec, k=k)
    dt = (time.time() - t0) * 1000
    print(f"k={k}  tiempo={dt:.1f} ms  top1={res[0][2][:80].strip()}...")

print("\nTop 5 Milvus:")
for rank, (idx, score, text, meta) in enumerate(milvus_search(query_vec, k=5), 1):
    print(f"[{rank}] id={idx} score={score:.4f} :: {text[:120].strip()}...")

k=5  tiempo=17218.5 ms  top1=Battery tester A battery tester is an electronic device intended for testing the...
k=20  tiempo=10206.0 ms  top1=Battery tester A battery tester is an electronic device intended for testing the...

Top 5 Milvus:
[1] id=10176 score=0.1297 :: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
[2] id=1 score=0.1382 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
[3] id=10177 score=0.1599 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
[4] id=71872 score=0.1614 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...
[5] id=37409 score=0.1655 :: shorting the measurement points together and performing an adjustment for zero ohms indication prior to each measurement.

### Respuestas a las preguntas de Milvus

**¿Qué parámetros ajusté para precisión vs velocidad?** En Milvus Lite el índice por defecto es `AUTOINDEX`, que internamente monta un HNSW. Para tocar el balance se puede pasar `index_params={"index_type": "HNSW", "params": {"M": 16, "efConstruction": 200}}` al crear la colección, y en búsqueda `search_params={"ef": 64}`. `ef` alto = más precisión, más lento. `ef` bajo = más rápido, menos preciso.

**¿Evidencia de que ANN cambia resultados?** Comparando el top-k de Milvus contra FAISS `IndexFlat` (que es exacto), si hay diferencias en los IDs, esa es la evidencia. En corpus grandes suele haber 1 o 2 diferencias en el top 10. En este notebook los tamaños son chicos, por eso ANN suele coincidir 100% con exacto. Con millones de vectores la diferencia aparece.

*Dato curioso de métodos numéricos:* HNSW es un grafo jerárquico donde cada capa es una versión "más rala" del grafo de abajo. Bajás piso por piso hasta llegar al vecino. Es como binary search pero en espacio vectorial. La construcción es cara, la búsqueda es logarítmica.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [18]:
!pip install "weaviate-client>=4.0" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle 2.0.2 requires kagglesdk<1.0,>=0.1.20, which is not installed.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


**Aviso sobre Weaviate:** el modo embebido de Weaviate baja un binario y lo corre local. Funciona bien en Linux/Mac. En Windows a veces falla y hay que levantar un contenedor Docker:

```
docker run -d -p 8080:8080 -p 50051:50051 --name weaviate semitechnologies/weaviate:latest
```

El código que sigue prueba primero embedded, si no arranca cae al cliente HTTP local.

In [19]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery

# Intento embedded, si no anda uso HTTP local (docker)
try:
    wclient = weaviate.connect_to_embedded()
except Exception:
    wclient = weaviate.connect_to_local()

# Recreo la colección desde cero
if wclient.collections.exists("Document"):
    wclient.collections.delete("Document")

wclient.collections.create(
    name="Document",
    vectorizer_config=Configure.Vectorizer.none(),  # traigo mis propios vectores
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="doc_id", data_type=DataType.INT),
    ],
)

col = wclient.collections.get("Document")

# Inserto con batch para acelerar
with col.batch.dynamic() as batch:
    for i in range(len(embeddings)):
        batch.add_object(
            properties={
                "text": chunks_df.iloc[i]["text"],
                "doc_id": int(chunks_df.iloc[i]["doc_id"]),
            },
            vector=embeddings[i].tolist(),
        )

print("Weaviate listo")

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 12765


Weaviate listo


In [20]:
def weaviate_search(query_embedding, k=5):
    res = col.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        return_metadata=MetadataQuery(distance=True),
    )
    salida = []
    for obj in res.objects:
        salida.append((
            str(obj.uuid),
            obj.metadata.distance,
            obj.properties["text"],
            {"doc_id": obj.properties["doc_id"]}
        ))
    return salida

# Ejemplo con k=5
print(f"Query: {query_text}\n")
for rank, (idx, dist, text, meta) in enumerate(weaviate_search(query_vec, k=5), 1):
    print(f"[{rank}] dist={dist:.4f} :: {text[:120].strip()}...")

wclient.close()

Query: Battery measuring

[1] dist=0.1297 :: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
[2] dist=0.1382 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
[3] dist=0.1599 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
[4] dist=0.1609 :: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply...
[5] dist=0.1614 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


### Respuestas de Weaviate

**Schema + objetos vs tabla + filas.** En una tabla SQL definís columnas con tipos y las filas son valores. En Weaviate definís una "clase" con propiedades tipadas y cada objeto tiene un UUID además de propiedades. La diferencia práctica: el objeto tiene identidad global (UUID) y viene pensado para tener un vector asociado como ciudadano de primera. En SQL el vector es una columna más.

**Trade-off complejidad vs expresividad.** Weaviate te obliga a pensar en el schema desde el arranque (más setup) pero después podés hacer queries híbridas (vector + BM25 + filtros de metadata) en una sola llamada. En Chroma o Qdrant también se puede, pero Weaviate lo tiene más pulido cuando el schema es rico.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [21]:
!pip install chromadb --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [22]:
import chromadb

# Cliente en memoria (efímero)
chroma_client = chromadb.Client()

# Si la colección ya existe, la borro para empezar limpio
try:
    chroma_client.delete_collection("wiki")
except Exception:
    pass

col_chroma = chroma_client.create_collection(
    name="wiki",
    metadata={"hnsw:space": "cosine"},  # métrica coseno
)

# Chroma acepta todo en listas paralelas
BATCH = 1000
for i in range(0, len(embeddings), BATCH):
    end = min(i + BATCH, len(embeddings))
    col_chroma.add(
        ids=[str(j) for j in range(i, end)],
        embeddings=[embeddings[j].tolist() for j in range(i, end)],
        documents=[chunks_df.iloc[j]["text"] for j in range(i, end)],
        metadatas=[{"doc_id": int(chunks_df.iloc[j]["doc_id"])} for j in range(i, end)],
    )

print(f"Chroma listo con {col_chroma.count()} vectores")

Chroma listo con 79104 vectores


In [23]:
def chroma_search(query_embedding, k=5):
    res = col_chroma.query(
        query_embeddings=[query_embedding[0].tolist()],
        n_results=k,
    )
    salida = []
    for i in range(len(res["ids"][0])):
        salida.append((
            res["ids"][0][i],
            res["distances"][0][i],
            res["documents"][0][i],
            res["metadatas"][0][i]
        ))
    return salida

print(f"Query: {query_text}\n")
for rank, (idx, dist, text, meta) in enumerate(chroma_search(query_vec, k=5), 1):
    print(f"[{rank}] id={idx} dist={dist:.4f} :: {text[:120].strip()}...")

Query: Battery measuring

[1] id=10176 dist=0.1297 :: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
[2] id=1 dist=0.1382 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
[3] id=10177 dist=0.1599 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
[4] id=37406 dist=0.1609 :: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply...
[5] id=71872 dist=0.1614 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


### Respuestas de Chroma

**¿Comparado con Qdrant/Milvus?** Mucho más simple. Chroma es literalmente 4 líneas para crear colección + insertar + consultar. No hay schema, no hay tipos, no hay configuración de índice. Ideal para prototipar RAG en un fin de semana.

**Limitaciones en producción.** No escala tan bien horizontalmente (no está pensado para clusters distribuidos como Milvus). La versión efímera pierde todo al cerrar el proceso, hay que usar `PersistentClient` para persistir. Los filtros por metadata son más limitados que en Qdrant o Weaviate. Y no tiene el nivel de tuning de índice que sí tiene Milvus.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


**Aviso sobre pgvector:** requiere PostgreSQL con la extensión `pgvector` instalada. La forma rápida es Docker:

```
docker run -d --name pg-vec -e POSTGRES_PASSWORD=admin -p 5432:5432 pgvector/pgvector:pg16
```

Si no tenés Docker, el código de abajo también viene en una versión con **SQLite + sqlite-vec**, que corre local sin servidor y muestra la misma idea: guardar el vector como columna y hacer `ORDER BY distancia LIMIT k`.

In [24]:
!pip install psycopg2-binary pgvector sqlite-vec --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 18.7 MB/s eta 0:00:00


#### Versión SQLite + sqlite-vec (sin servidor)

In [25]:
import sqlite3
import sqlite_vec
import struct

def serialize_vec(v):
    """SQLite guarda vectores como blobs de floats little-endian."""
    return struct.pack(f"{len(v)}f", *v)

# Abro DB en memoria y cargo la extensión sqlite-vec
conn = sqlite3.connect(":memory:")
conn.enable_load_extension(True)
sqlite_vec.load(conn)
conn.enable_load_extension(False)

D = embeddings.shape[1]

# Tabla virtual con soporte de vectores
conn.execute(f"CREATE VIRTUAL TABLE documents USING vec0(id INTEGER PRIMARY KEY, embedding float[{D}])")
# Tabla normal para el texto y metadata
conn.execute("CREATE TABLE docs_meta (id INTEGER PRIMARY KEY, text TEXT, doc_id INTEGER)")

# Inserto todo
for i in range(len(embeddings)):
    conn.execute("INSERT INTO documents(id, embedding) VALUES (?, ?)",
                 (i, serialize_vec(embeddings[i].tolist())))
    conn.execute("INSERT INTO docs_meta(id, text, doc_id) VALUES (?, ?, ?)",
                 (i, chunks_df.iloc[i]["text"], int(chunks_df.iloc[i]["doc_id"])))

conn.commit()
print("SQLite-vec listo")

SQLite-vec listo


In [26]:
def pgvector_search(query_embedding, k=5):
    q_blob = serialize_vec(query_embedding[0].tolist())
    # Uso JOIN clásico entre la tabla vectorial y la de metadata
    rows = conn.execute("""
        SELECT documents.id, documents.distance, docs_meta.text, docs_meta.doc_id
        FROM documents
        JOIN docs_meta ON documents.id = docs_meta.id
        WHERE documents.embedding MATCH ?
        AND k = ?
        ORDER BY distance
    """, (q_blob, k)).fetchall()
    return [(r[0], r[1], r[2], {"doc_id": r[3]}) for r in rows]

print(f"Query: {query_text}\n")
for rank, (idx, dist, text, meta) in enumerate(pgvector_search(query_vec, k=5), 1):
    print(f"[{rank}] id={idx} dist={dist:.4f} :: {text[:120].strip()}...")

Query: Battery measuring

[1] id=10176 dist=0.5092 :: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
[2] id=1 dist=0.5257 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
[3] id=10177 dist=0.5655 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
[4] id=37406 dist=0.5672 :: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply...
[5] id=71872 dist=0.5682 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...


#### Referencia: cómo sería lo mismo con PostgreSQL + pgvector

```python
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(host="localhost", user="postgres", password="admin", dbname="postgres")
register_vector(conn)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
cur.execute(f"""
    CREATE TABLE documents (
        id INT PRIMARY KEY,
        text TEXT,
        doc_id INT,
        embedding vector({D})
    )
""")

for i in range(len(embeddings)):
    cur.execute("INSERT INTO documents VALUES (%s, %s, %s, %s)",
                (i, chunks_df.iloc[i]["text"], int(chunks_df.iloc[i]["doc_id"]), embeddings[i].tolist()))
conn.commit()

# Búsqueda: <=> es la distancia coseno en pgvector
cur.execute("SELECT id, text, embedding <=> %s AS distance FROM documents ORDER BY distance LIMIT %s",
            (query_vec[0].tolist(), 5))
for row in cur.fetchall():
    print(row)
```

### Respuestas de pgvector

**Explicabilidad.** Muy alta. Es SQL de toda la vida más un operador nuevo (`<=>` para coseno, `<->` para L2). Cualquier persona que sepa SQL entiende la query. En Qdrant o Weaviate se ve más "mágico" porque la API esconde qué hace por debajo.

**Ventajas del mundo SQL.** JOINs con otras tablas del negocio (usuarios, productos, permisos), transacciones ACID, agregaciones (`GROUP BY`, `COUNT`), filtros con `WHERE` sobre columnas comunes. Todo eso ya existe. Si tu app ya vive en Postgres, agregar búsqueda semántica es solo instalar una extensión.

**Limitaciones frente a bases dedicadas.** Postgres no fue diseñado para escalar horizontalmente en búsqueda de vecinos. Con millones de embeddings de 768+ dimensiones empieza a sufrir. Milvus o Qdrant en cluster escalan mejor. También, los índices ANN de pgvector (HNSW, IVFFlat) son buenos pero no tan afinados como los de las dedicadas.

## Parte 8: Experimentación

Ya con las 6 herramientas armadas, quiero comparar cosas que no pedía el cuaderno.

### 8.1 Comparar velocidad de búsqueda entre motores
### 8.2 Ver si los top 5 coinciden entre motores (overlap)
### 8.3 Probar filtro por metadata en Qdrant
### 8.4 Query en español a un corpus en inglés (E5 es multilingüe? no realmente, veamos qué pasa)

In [31]:
# 8.1 Benchmark de tiempo: 5 queries por motor, promedio en ms
import time

motores = {
    "FAISS":  lambda q, k: index_faiss.search(q, k),
    "Qdrant": lambda q, k: qdrant_search(q, k),
    "Milvus": lambda q, k: milvus_search(q, k),
    "Chroma": lambda q, k: chroma_search(q, k),
    "SQLite-vec": lambda q, k: pgvector_search(q, k),
}

queries_test = [
    "battery indicator problems",
    "history of solar energy",
    "computer graphics techniques",
    "military communications",
    "space exploration NASA",
]
query_vecs = [embed_query(q) for q in queries_test]

resultados_tiempo = {}
for nombre, fn in motores.items():
    try:
        t0 = time.time()
        for qv in query_vecs:
            _ = fn(qv, 5)
        dt = (time.time() - t0) / len(query_vecs) * 1000
        resultados_tiempo[nombre] = dt
    except Exception as e:
        print(f"Error en motor {nombre}: {e}")

print("\nVelocidad promedio por motor (ms/query):")
for k, v in sorted(resultados_tiempo.items(), key=lambda x: x[1]):
    print(f"{k:12s} {v:8.2f} ms")


Velocidad promedio por motor (ms/query):
Chroma           4.28 ms
FAISS           20.53 ms
SQLite-vec     124.84 ms
Qdrant         365.82 ms
Milvus        4318.53 ms


In [32]:
# 8.2 Overlap de resultados: ¿los motores traen los mismos top 5?
q = embed_query("battery indicator problems")

# FAISS: los IDs son los índices directos
_, I_faiss = index_faiss.search(q, 5)
top_faiss = set(int(i) for i in I_faiss[0])

top_qdrant = set(r[0] for r in qdrant_search(q, 5))
top_milvus = set(r[0] for r in milvus_search(q, 5))
top_chroma = set(int(r[0]) for r in chroma_search(q, 5))
top_sqlite = set(r[0] for r in pgvector_search(q, 5))

print("FAISS ∩ Qdrant :", len(top_faiss & top_qdrant), "/ 5")
print("FAISS ∩ Milvus :", len(top_faiss & top_milvus), "/ 5")
print("FAISS ∩ Chroma :", len(top_faiss & top_chroma), "/ 5")
print("FAISS ∩ SQLite :", len(top_faiss & top_sqlite), "/ 5")
print()
print("Todos coinciden en:", len(top_faiss & top_qdrant & top_milvus & top_chroma & top_sqlite), "/ 5")

FAISS ∩ Qdrant : 5 / 5
FAISS ∩ Milvus : 5 / 5
FAISS ∩ Chroma : 5 / 5
FAISS ∩ SQLite : 5 / 5

Todos coinciden en: 5 / 5


In [33]:
# 8.3 Filtro por metadata en Qdrant: solo chunks del doc_id 5
from qdrant_client.models import Filter, FieldCondition, MatchValue

q = embed_query("science")
# Usamos query_points con el filtro correspondiente
res = qdrant.query_points(
    collection_name="wiki",
    query=q[0].tolist(),
    limit=3,
    query_filter=Filter(must=[FieldCondition(key="doc_id", match=MatchValue(value=5))])
)

print("Solo chunks del documento 5:")
for h in res.points:
    print(f"  score={h.score:.4f} doc_id={h.payload['doc_id']} :: {h.payload['text'][:100].strip()}...")

Solo chunks del documento 5:
  score=0.7202 doc_id=5 :: Capacity loss Capacity loss or capacity fading is a phenomenon observed in rechargeable battery usag...


In [34]:
# 8.4 Query en español a un corpus en inglés
# E5 base v2 es SOLO inglés. Aquí veremos que no encuentra bien.
query_es = "problemas con la batería del auto"
qv_es = embed_query(query_es)

print(f"Query en español: '{query_es}'\n")
print("Top 3 con E5-base-v2 (solo entrenado en inglés):")
# Usamos la función qdrant_search que ya está corregida
for rank, (idx, score, text, meta) in enumerate(qdrant_search(qv_es, k=3), 1):
    print(f"[{rank}] score={score:.4f} :: {text[:120].strip()}...")

print("\nComparación: la misma query en inglés")
query_en = "car battery problems"
qv_en = embed_query(query_en)
for rank, (idx, score, text, meta) in enumerate(qdrant_search(qv_en, k=3), 1):
    print(f"[{rank}] score={score:.4f} :: {text[:120].strip()}...")

Query en español: 'problemas con la batería del auto'

Top 3 con E5-base-v2 (solo entrenado en inglés):
[1] score=0.8034 :: ffort from the driverâ€™s gear selection. There have been safety issues identified with production vehicles implementing...
[2] score=0.8029 :: JuÃ¡rez, NezahualcÃ³yotl, NicolÃ¡s Romero, TecÃ¡mac, Tlalnepantla de Baz, TultitlÃ¡n and Valle de Chalco Solidaridad. Ho...
[3] score=0.7879 :: or the virtual elimination of hunting is a straight track, with an attendant right-of-way problem and incompatibility wi...

Comparación: la misma query en inglés
[1] score=0.8324 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
[2] score=0.8304 :: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...
[3] score=0.8239 :: hat the concept of a float voltage does not apply equally to all battery chemistries. For instance, lithium ion cells

### Notas del experimento 8.4

E5 base v2 solo fue entrenado con datos en inglés. Cuando le paso una query en español, el vector cae en una zona rara del espacio y la similitud contra los pasajes (todos en inglés) baja mucho. Los resultados igual son peores que con la misma query en inglés.

Si quisiera búsqueda multilingüe habría que cambiar a `intfloat/multilingual-e5-base`, que sí fue entrenado en varios idiomas y mapea "car" y "auto" a puntos cercanos en el espacio.

*Dato curioso de machine learning:* este es el famoso problema del *domain gap*. Un modelo aprende solo del dominio que vio. Como en clasificadores de imágenes que fallan cuando les cambias la iluminación. Los embeddings son igual: dependen de qué idioma, qué estilo, qué distribución de datos vieron durante el entrenamiento.

## Cierre

Recorrí seis herramientas para el mismo problema: FAISS, Qdrant, Milvus, Weaviate, Chroma y pgvector/SQLite-vec. Cada una tiene su lugar. FAISS para investigación pura. Qdrant y Milvus cuando querés metadata + escala. Weaviate cuando el schema es rico. Chroma para prototipos rápidos. pgvector cuando ya tenés Postgres. Al final todas resuelven la misma pregunta: dado un vector, cuáles son los K más parecidos en el corpus.